# Magnetic ordering

A composition does not determine a structure, and a structure does not determine
a magnetic state. Iron, nickel, cobalt and most of their compounds have several
spin arrangements at nearly the same energy, and which one is lowest changes the
formation energy by tenths of an eV per atom — comfortably enough to move a
material on or off the convex hull.

So a stability screen over magnetic materials has a step before it that a screen
over oxides of aluminium does not: enumerate the orderings, and find out which
one you are actually computing.

This is the shortest tutorial here, and it ends by refusing to answer the
question. That refusal is the point.

In [1]:
import matverse as mv
import numpy as np
import pandas as pd

mv.pl.set_style()

🔬 Starting plot initialization...
🧪 Calculators available: 6
    • emt — EMT (LGPL-2.1)
    • lj — Lennard-Jones (LGPL-2.1)
    • mace-mpa — mace-mpa (unstated)
    • mace-omat — mace-omat (unstated)
    • sevennet — sevennet (unstated)
    • chgnet — chgnet (unstated)


🖥️ NVIDIA CUDA GPUs: 1
    • [CUDA 0] NVIDIA H100 80GB HBM3 — 79.1 GB, compute 9.0

                   __
   ____ ___  ____ _/ /__   _____  _____________
  / __ `__ \/ __ `/ __/ | / / _ \/ ___/ ___/ _ \
 / / / / / / /_/ / /_ | |/ /  __/ /  (__  )  __/
/_/ /_/ /_/\__,_/\__/ |___/\___/_/  /____/\___/

🔖 Version: 0.1.18   🧮 Functions: 155   📚 Tutorials: https://matverse.readthedocs.io/
✅ set_style complete.



## Loading a dataset

Three materials chosen to span the cases: a magnetic element, a magnetic alloy,
and something with no magnetic ion at all.

In [2]:
from pymatgen.core import Lattice, Structure


def fcc(symbol, a):
    return Structure(Lattice.cubic(a), [symbol] * 4,
                     [[0, 0, 0], [0, .5, .5], [.5, 0, .5], [.5, .5, 0]])


md = mv.data.from_structures([
    fcc("Ni", 3.524),
    Structure(Lattice.cubic(3.57), ["Ni", "Ni", "Ni", "Al"],
              [[0, 0, 0], [0, .5, .5], [.5, 0, .5], [.5, .5, 0]]),
    fcc("Al", 4.050),
])
mv.pp.standardize(md)
mv.pp.describe(md)

md.obs[["formula", "spacegroup", "nsites"]]

,formula,spacegroup,nsites
0,Ni,Fm-3m,4
1,AlNi3,Pm-3m,4
2,Al,Fm-3m,4


## Which elements can even carry a moment

In [3]:
mv.mag.describe(md)

md.obs[["formula", "magnetic_order", "n_magnetic_species",
        "total_magmom", "absolute_magmom"]].round(3)

,formula,magnetic_order,n_magnetic_species,total_magmom,absolute_magmom
0,Ni,unknown,1,NaN,NaN
1,AlNi3,unknown,1,NaN,NaN
2,Al,unknown,0,NaN,NaN


Read those two columns carefully, because they answer different questions.

`magnetic_order` is `unknown` for all three, and the moments are NaN. That is
correct: `mv.mag.describe` reports what the **structure carries**, and a CIF or
a hand-built cell carries no moments at all. It is not saying nickel is
non-magnetic; it is saying nobody has told this structure anything.

`n_magnetic_species` is the column doing the work here, and it comes from the
chemistry rather than from the file — 1 for the two nickel-bearing materials, 0
for aluminium. That is what decides whether enumerating orderings is even a
meaningful thing to do.

The set being tested against is `mv.mag.MAGNETIC_ELEMENTS`: the 3d, 4d and 5d
transition metals plus the lanthanides and actinides.

In [4]:
sorted(mv.mag.MAGNETIC_ELEMENTS)[:20]

['Ce',
 'Co',
 'Cr',
 'Cu',
 'Dy',
 'Er',
 'Eu',
 'Fe',
 'Gd',
 'Ho',
 'Mn',
 'Nd',
 'Ni',
 'Np',
 'Pr',
 'Pu',
 'Sm',
 'Tb',
 'Ti',
 'Tm']

## Enumerating orderings

`mv.mag.orderings` returns a **new object** whose rows are spin configurations,
with `obs['parent']` pointing back at the material — the same shape as
`mv.pp.defects` and `mv.surf.slabs`.

In [5]:
orderings = mv.mag.orderings(md, max_orderings=6)
orderings

AnnData object with n_obs × n_vars = 8 × 2
    obs: 'parent', 'ordering', 'ordering_index', 'total_magmom', 'is_magnetic'
    var: 'Z', 'atomic_mass', 'atomic_radius', 'electronegativity', 'group', 'period', 'melting_point', 'boiling_point', 'molar_volume', 'thermal_conductivity', 'electrical_resistivity', 'average_ionic_radius', 'max_oxidation_state', 'min_oxidation_state', 'is_metal', 'is_transition_metal', 'is_alkali', 'is_alkaline', 'is_metalloid', 'is_halogen', 'is_noble_gas', 'is_chalcogen', 'is_lanthanoid', 'is_actinoid', 'is_rare_earth_metal', 'block'
    uns: 'features', 'levels', 'provenance', 'X_is', 'magnetic_orderings'
    obsm: 'structures'

In [6]:
mv.pp.describe(orderings)
orderings.obs[["parent", "formula", "ordering", "ordering_index",
               "total_magmom", "is_magnetic"]].round(3)

,parent,formula,ordering,ordering_index,total_magmom,is_magnetic
0,0,Ni,fm,0,20.0,True
1,0,Ni,fim,1,10.0,True
2,0,Ni,afm,2,0.0,True
3,0,Ni,fim,3,-10.0,True
4,1,AlNi3,fm,0,15.0,True
5,1,AlNi3,fim,1,5.0,True
6,1,AlNi3,fim,2,-5.0,True
7,2,Al,nonmagnetic,0,0.0,False


Three things worth reading in that table.

The ferromagnetic ordering has the largest total moment, and the
antiferromagnetic one has **exactly zero** — that is what antiferromagnetic
means, and it is a check rather than a coincidence.

Aluminium comes through as a single `nonmagnetic` row rather than being dropped.
Every input material is still represented, so nothing needs re-joining by hand
afterwards.

And the moments are on the structures themselves, as a site property, so they
survive being written to disk and reach any calculator that knows what to do
with them.

In [7]:
mv.structures(orderings)[0].site_properties["magmom"]

[5.0, 5.0, 5.0, 5.0]

```{note}
pymatgen's antiferromagnetic enumeration calls out to **enumlib**, which is not
pip-installable and is absent from most environments. matverse falls back to a
simpler construction when it is missing — and records that it did.

Falling back is fine. Falling back silently is not: the fallback explores fewer
configurations, so a ground state found with it is a weaker claim than one found
with enumlib, and a reader has to be able to tell which they are looking at.
```

In [8]:
orderings.uns["magnetic_orderings"]["errors"]

["0: RuntimeError: EnumlibAdaptor requires the executables 'enum.x' or 'multienum.x' and 'makestr.x' or 'makeStr.py' to be in the path. Please download the library at https://github.com/msg-byu/enumlib and follow the instructions in the README to compile these two executables accordingly.; used the built-in fallback",
 "1: RuntimeError: EnumlibAdaptor requires the executables 'enum.x' or 'multienum.x' and 'makestr.x' or 'makeStr.py' to be in the path. Please download the library at https://github.com/msg-byu/enumlib and follow the instructions in the README to compile these two executables accordingly.; used the built-in fallback"]

## Picking the ground state

Compute every ordering, and let the lowest energy win. The winner's energy lands
back on the parent material.

In [9]:
mv.calc.energy(orderings, level="emt")
mv.mag.ground_state(orderings, md, level="emt")

md.obs[["formula", "magnetic_ordering_emt", "magnetic_spread_emt",
        "energy_per_atom_emt"]].round(6)

,formula,magnetic_ordering_emt,magnetic_spread_emt,energy_per_atom_emt
0,Ni,fm,0.0,-0.007592
1,AlNi3,fm,0.0,0.270962
2,Al,nonmagnetic,NaN,-0.001502


## The spread is zero, and that is the answer

`magnetic_spread` is the energy range across the orderings of one material, and
here it is **exactly zero** for both magnetic materials.

That is not a bug. EMT has no notion of spin at all, so every ordering of nickel
is the same set of atoms in the same positions and gets the same energy. The
calculator cannot distinguish them, and the object says so in a number rather
than by producing a confident arbitrary answer.

Which is the useful behaviour: `magnetic_ordering_emt` says `fm` for nickel, and
`magnetic_spread_emt` says that claim is worth nothing.

In [10]:
md.uns["magnetic"]["emt"]

{'n_with_alternatives': 2,
 'max_spread': 0.0,
 'collinear': True,
 'note': 'magnetic_spread is the gap between the best and worst ordering; a large one means the hull depends on this choice'}

```{warning}
**This is where the tutorial stops, because the calculator that ships with
matverse cannot go further.** Resolving a magnetic ground state needs a
spin-polarised method — DFT with initialised moments, or a machine-learned
potential trained on magnetic configurations.

The enumeration above is real and reusable. The energies are not.
```

With such a calculator registered, the rest is unchanged:

```python
from mace.calculators import mace_mp

mv.calc.register_calculator("mace-mpa", lambda: mace_mp(model="medium-mpa-0"),
                            kind="mlip", method="MACE-MPA-0",
                            reference="PBE+U", license="MIT")

orderings = mv.mag.orderings(md, max_orderings=8)
mv.calc.relax(orderings, level="mace-mpa")
mv.mag.ground_state(orderings, md, level="mace-mpa")
```

or by writing the orderings out for DFT:

```python
mv.dft.write_inputs(orderings, code='vasp', preset='relax',
                    directory='orderings/')
```

## Why this belongs before the hull

The energy `mv.mag.ground_state` writes back is under the **ordinary** column
name the calculator would have produced — `energy_per_atom_emt`, not
`energy_per_atom_emt_magnetic`.

In [11]:
[c for c in md.obs.columns if c.startswith("energy")]

['energy_per_atom_emt']

That is deliberate. `mv.thermo.hull` needs no special case for magnetism: it
sees a normal energy column that happens to be the magnetic ground state, and a
pipeline written without magnetism in mind keeps working when magnetism is
added in front of it.

The alternative — a specially-named column — would mean every downstream
function needed to know about magnetic ordering, and the ones that did not know
would silently use the wrong energy.

```{seealso}
[Screening, end to end](screening.ipynb) is the pipeline this step goes in front
of. [Defects and diffusion](defects_and_diffusion.ipynb) has the same shape:
enumerate configurations, compute them all, let the object record which won.
```